# Module 22 — Short-term memory: full vs window vs summary

**THE ONE IDEA:** module 02 said *memory is the list*. That list grows without bound, so
eventually you must **throw some of it away**. The three ways of doing that fail
differently — and the point of this module is to watch **what each one forgets.**

| strategy | keeps | forgets |
|---|---|---|
| **full** | everything | nothing — until you hit the context limit and cost |
| **window** | last N messages | anything older, **completely and abruptly** |
| **summary** | a précis + last N | *detail*, gradually — and it can summarise wrongly |

A fact is planted in turn 1 and asked for in turn 8. Only one strategy will still have it.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
from _providers import get_client

client, MODEL, _ = get_client("openai")

def ask(messages, max_tok=200):
    r = client.chat.completions.create(model=MODEL, max_tokens=max_tok, messages=messages)
    return r.choices[0].message.content.strip(), r.usage.prompt_tokens

# The planted fact is in turn 1. The probe is turn 8.
TURNS = ["My client reference is MX-7741 and she wants a 5-year fix.",
         "What is the standard variable rate roughly?",
         "And the typical early repayment charge in year 2?",
         "Is a 90% LTV realistic for a first-time buyer?",
         "What documents prove income for the self-employed?",
         "How long do these applications usually take?",
         "What happens if she overpays by 10% a year?"]
PROBE = "What was my client's reference number?"

## Strategy 1 — full history

In [ ]:
def run_full():
    msgs, costs = [], []
    for t in TURNS + [PROBE]:
        msgs.append({"role": "user", "content": t})
        out, tin = ask(msgs)
        msgs.append({"role": "assistant", "content": out})
        costs.append(tin)
    return out, costs

ans_full, cost_full = run_full()
print("FULL   ->", ans_full[:130])
print("input tokens per turn:", cost_full)

## Strategy 2 — sliding window of the last N messages

Cheap and trivially correct to implement. It forgets **abruptly**: the moment turn 1
slides out, the reference is gone with no warning and no degradation.

In [ ]:
def run_window(keep=6):
    msgs, costs = [], []
    for t in TURNS + [PROBE]:
        msgs.append({"role": "user", "content": t})
        out, tin = ask(msgs[-keep:])          # <- only the tail is ever sent
        msgs.append({"role": "assistant", "content": out})
        costs.append(tin)
    return out, costs

ans_win, cost_win = run_window()
print("WINDOW ->", ans_win[:130])
print("input tokens per turn:", cost_win)

## Strategy 3 — running summary plus a short tail

Old turns are compressed into a paragraph rather than deleted. Detail is lost
**gradually**, and whether the reference survives depends on whether the summariser
judged it important.

In [ ]:
def run_summary(keep=4):
    msgs, summary, costs = [], "", []
    for t in TURNS + [PROBE]:
        msgs.append({"role": "user", "content": t})
        if len(msgs) > keep:
            old, msgs = msgs[:-keep], msgs[-keep:]
            summary, _ = ask([{"role": "user", "content":
                "Update this running summary. Keep ALL identifiers, names and numbers.\n"
                f"SUMMARY SO FAR: {summary}\nNEW TURNS: {old}"}], max_tok=220)
        ctx = ([{"role": "system", "content": f"Conversation so far: {summary}"}] + msgs
               if summary else msgs)
        out, tin = ask(ctx)
        msgs.append({"role": "assistant", "content": out})
        costs.append(tin)
    return out, costs, summary

ans_sum, cost_sum, summary = run_summary()
print("SUMMARY ->", ans_sum[:130])
print("\nthe summary that carried the context:\n ", summary[:300])

## Side by side — what each one forgot

In [ ]:
def kept(a): return "MX-7741" in a or "MX7741" in a.replace("-", "")

print(f"{'strategy':10} {'kept MX-7741':>13} {'final turn input':>17} {'total input':>12}")
print("-" * 56)
for name, ans, costs in [("full", ans_full, cost_full), ("window", ans_win, cost_win),
                         ("summary", ans_sum, cost_sum)]:
    print(f"{name:10} {str(kept(ans)):>13} {costs[-1]:>17} {sum(costs):>12}")

print("""
LESSON - there is no free compaction. You are choosing WHICH failure you get.

  full      remembers everything, and input grows every turn (module 11's curve).
            Fine for 10 turns. Fatal at 500, and you pay for it long before then.

  window    cheapest and trivially correct to implement. Forgets ABRUPTLY: the
            turn that slides out takes its facts with it, with no signal. The
            model does not say 'I forgot' - it answers confidently and wrongly.

  summary   forgets GRADUALLY, and adds a second failure mode: the summariser
            itself can drop or distort a detail. Note the prompt above has to BEG
            it to keep identifiers - that instruction is load-bearing, and it is
            still only a request.

Production usually layers them: a window for recency, a summary for the middle,
and - for anything that must never be lost, like a client reference - a
STRUCTURED store outside the conversation entirely.

That structured store is module 23, and it is the tier most people skip on their
way to vectors.""")

---

**Next:** `23_memory_entity_keyvalue.ipynb`